# CoT-NAFNet on CDD-11-30

Import this notebook from GitHub, attach the two existing Kaggle inputs, enable a GPU, and run all cells. The default run performs only a read-only audit. Set `RUN_TRAIN = True` after the audit succeeds.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("CWD:", Path.cwd())

In [ ]:
from pathlib import Path
import torch

CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
print("GPU:", torch.cuda.get_device_name(0))
print("CDD-11:", CDD11_ROOT)
print("Pretrained:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Read-only audit: all pairs, split leakage, and exact compatibility of all four checkpoints.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit.json",
], check=True)

In [ ]:
# Safety switches. A freshly imported notebook audits inputs but does not start a long run.
RUN_TRAIN = False
RUN_EVALUATION = False
PRESET = "gopro32"  # gopro32 is the 17M primary model; other options: gopro64, sidd32, sidd64
EPOCHS = 100
OUTPUT_DIR = Path("/kaggle/working/experiments/cot_nafnet_gopro32_seed42")

In [ ]:
if RUN_TRAIN:
    command = [
        "python", "-m", "hybrid_cot_nafnet.train_kaggle",
        "--data-root", str(CDD11_ROOT),
        "--output-dir", str(OUTPUT_DIR),
        "--model", "hybrid",
        "--preset", PRESET,
        "--pretrained", "auto",
        "--epochs", str(EPOCHS),
        "--max-minutes", "0",
        "--crop-size", "256",
        "--batch-size", "2",
        "--microbatch-size", "1",
        "--patches-per-image", "2",
        "--num-workers", "2",
        "--seed", "42",
    ]
    subprocess.run(command, check=True)
else:
    print("Training skipped. Set RUN_TRAIN = True after reviewing audit.json.")

In [ ]:
if RUN_EVALUATION:
    checkpoint = OUTPUT_DIR / "best.pt"
    assert checkpoint.is_file(), f"Missing checkpoint: {checkpoint}"
    subprocess.run([
        "python", "-m", "hybrid_cot_nafnet.evaluate",
        "--checkpoint", str(checkpoint),
        "--data-root", str(CDD11_ROOT),
        "--output-dir", str(OUTPUT_DIR / "evaluation"),
        "--tile", "256", "--overlap", "32",
        "--num-workers", "2",
    ], check=True)
else:
    print("Evaluation skipped. Set RUN_EVALUATION = True after training.")